| <div> <img src="https://storage.googleapis.com/open-ff-common/openFF_logo.png" width="100"/><br>Open-FF</div>|<h1>Exploring fracking chemicals<br>within a selected watershed</h1>|<center><a href="https://www.fractracker.org/" title="FracTracker Alliance"><img src="https://storage.googleapis.com/open-ff-common/2025_FT_logo_icon.png" alt="FracTracker logo" width="75"><br>Sponsored by<br> FracTracker Alliance</a></center>|
|---|---|---|

In [ ]:
import sys
sys.path.insert(0,'c:/MyDocs/integrated/') # adjust to your setup
%run Explore_near_location_support.py

In [ ]:
cols = ['DisclosureId', 'JobEndDate', 'JobStartDate', 'OperatorName', 'APINumber',
        'TotalBaseWaterVolume','FFVersion', 'TVD', 'api10',
        'WellName', 'bgOperatorName', 'StateName', 'bgStateName','ingKeyPresent',
        'CountyName', 'bgCountyName', 'Latitude', 'bgLatitude', 'Longitude', 'bgLongitude',
        #'bgLocationSource', 'latlon_too_coarse', 'loc_name_mismatch', 'loc_within_state', 'loc_within_county',
        'date','ws_perc_total', 'no_chem_recs', #'is_duplicate', 'primarySupplier',
        #'MI_inconsistent', 'IngredientsId', 
        'CASNumber', 'IngredientName', 'PercentHFJob', 'Supplier', 'Purpose', 'TradeName', 
        #'PercentHighAdditive', 'MassIngredient', 'ingKeyPresent', 'reckey',
        'bgCAS', #'ingredCommonName', 'bgSupplier', 'dup_rec', 'cleanMI', 'calcMass', 'massComp', 'massCompFlag',
        'massSource', 
        'mass', 'bgIngredientName', 'is_on_CWA', 'is_on_DWSHA', 'is_on_AQ_CWA', 'is_on_HH_CWA', 
        'is_on_IRIS', 'is_on_PFAS_list', 'epa_pref_name', 'is_on_NPDWR', 'is_on_prop65', 'is_on_TEDX', 'is_on_diesel', 
        'is_on_UVCB', 'rq_lbs','in_std_filtered']


dffn = hndl.curr_data

df = pd.read_parquet(dffn,
                     columns=cols)
# print(list(df.columns))
print(f'Num recs in df: {len(df)}')
# print(f'STATES included: {df.bgStateName.unique().tolist()}')
gb1 = df.groupby('APINumber', as_index=False).size()
gb1['FF_disc'] = gb1.apply(lambda x: th.getFFLink(x),axis=1)
df = df.merge(gb1[['APINumber','FF_disc']], on='APINumber',how='left',validate='m:1')
gb2 = df.groupby('DisclosureId',as_index=False)['APINumber'].first()
gb2['disc_link'] = gb2.apply(lambda x: th.getDisclosureLink(x.APINumber,x.DisclosureId,use_remote=True),axis=1)
df = df.merge(gb2[['DisclosureId','disc_link']], on='DisclosureId',how='left',validate='m:1')
df = df[df.in_std_filtered]

In [ ]:
# make wells gdf:

wellgdf = maps.make_as_well_gdf(df)

In [ ]:
lat_lon = '40.38415163662122, -79.62416933177497'  #PA GAIA?
lat_lon = '40.715187,-104.028656' # ECMC Martin 3031
lat_lon = '40.260464, -104.89143' # ECMC Swartz, Kerr McGEE
lat_lon = '40.461091726932736, -79.59789190158254'  # Beaver Run Reservoir - Athena pad
lat_lon = '40.489191654032055, -79.556930772574' # Beaver Run Reservoir
# lat_lon = '40.120052, -104.50129' # Verdad  "Arnold" well pad
# lat_lon = '39.653515, -108.17839' # Caerus "NPM M33" well pad in Garfield county , CO
# lat_lon = '40.26812 , -104.89382' # new Anadarko Camenisch 10-33HZ
# lat_lon = '40.631466 , -104.2226' # ancond pad of Bison
# lat_lon = '40.11137461329927, -81.37047635338446' # fire near Salt Fork Park?
# lat_lon = '32.41669138869378, -104.21467347364316' # NM near Carlsbad
# lat_lon = '40.120052,  -104.50129' # aronld pad in CO
# lat_lon = '39.61197314152577, -104.66510294280317' # Aurora Reservoir, CO
# lat_lon = '40.51648896110052, -80.43574155059288' # Raccoon Creek, PA; use scale = 10

# radius_input = 2000

from shapely.geometry import Point
ll_lst = lat_lon.split(',')
# Your coordinates (longitude, latitude)
longitude = float(ll_lst[1].strip())
latitude = float(ll_lst[0].strip())

# Create the Point object
point = Point(longitude, latitude)
# print(lat_lon,longitude, latitude, point,'\n\n')

# Create a small buffer around the point to define the bounding box
# A buffer of 0.1 degrees is ~6.9 miles, which is a reasonable search area
buffer = 0.1 
bbox = (longitude - buffer, latitude - buffer, longitude + buffer, latitude + buffer)

In [ ]:
huc_scale = 10 # should be 2,3,4,6,8,10 or 12

In [ ]:
# get watershed df for determining which sheds to plot
import geopandas as gpd
final_crs = 4326 # WGS84
gdb_path =  r"C:\MyDocs\integrated\ext_data\WBD_National_GDB.zip"
gdb_path = r"C:\MyDocs\integrated\ext_data\WBD_National_GPKG\WBD_National_GPKG.gpkg"
# gdb_path = "https://storage.googleapis.com/open-ff-common/ext_data/WBD_National_GPKG.gpkg"
huc_layer_name = 'WBDHU'+str(huc_scale)  # Example layer name, use the one you found
huc_colname = 'huc'+str(huc_scale)

print(f'Selected watershed scale: {huc_layer_name}')
print(f'Loading watershed database, may take a while for fine scale selection')

# Read the specific layer into a GeoDataFrame
# os.environ['GDAL_HTTP_UNSAFE_SSL'] = 'YES'
huc_gdf = gpd.read_file(gdb_path, layer=huc_layer_name,bbox=bbox)
huc_gdf = huc_gdf.drop(['loaddate','tnmid', 'metasourceid', 'sourcedatadesc', 'sourceoriginator',
                        'sourcefeatureid', 'referencegnis_ids', 'areaacres', 'areasqkm', 'globalid',
                        'shape_Length', 'shape_Area'],axis=1)
huc_gdf = huc_gdf.to_crs(final_crs)
# huc_gdf.columns

# huc_gdf.to_parquet(os.path.join(hndl.sandbox_dir,'huc8_gdf.parquet'))

# huc_gdf = gpd.read_parquet(os.path.join(hndl.sandbox_dir,'huc8_gdf.parquet'))
# huc8_gdf['huc8_name'] = huc8_gdf['name']
# # print(huc8_gdf.columns)

In [ ]:
# Find the watershed for the given point
# Assuming your GeoDataFrame is named 'watersheds_gdf'
containing_watershed = huc_gdf[huc_gdf.geometry.contains(point)]
containing_watershed = containing_watershed.to_crs(final_crs)
watershed_name = containing_watershed['name'].iloc[0]
print(watershed_name)

In [ ]:
# lat,lon = process_lat_lon_input(lat_lon)
display(maps.show_simple_map_and_shape(latitude,longitude,include_shape=True,
                                       area_df=containing_watershed,
                                       height=600,width=600))


In [ ]:
# radius_in_feet = process_radius_input(radius_input)
apilst = maps.find_wells_within_area(containing_watershed,
                                     wellgdf, buffer_m=0)
print(f'Number of well locations within watershed: {len(apilst)}')

ws_df = df[df.api10.isin(apilst)].copy()
print(f' -- total rcords: {len(ws_df)}')

In [ ]:
# # gather disclosureIds and filter apis
# min_year = 2022 
# max_year = 2027

# filt = (
#     print('Filter by Year')
#     dgb = dgb[dgb.DisclosureId.isin(disclst)].copy()
#     apis = dgb.api10.unique().tolist()
#     print(f'Pre-filter number of disclosures: {len(disclst)}, APIs: {len(apis)}')
#     didlst = dgb[dgb.date.dt.year>=min_year].DisclosureId.unique().tolist()
#     apis = dgb[dgb.DisclosureId.isin(didlst)].api10.unique().tolist()
#     print(f'Post-filter number of disclosures: {len(didlst)}, APIs: {len(apis)}')
#     t = df[df.DisclosureId.isin(didlst)]
#     dgb = dgb[dgb.DisclosureId.isin(didlst)]



In [ ]:
#show proprietary records
print(len(ws_df[ws_df.bgCAS=='proprietary'][['api10','bgCAS','mass','CASNumber','IngredientName']]))
gb = ws_df[ws_df.bgCAS=='proprietary'].groupby(['IngredientName'],as_index=False)['mass'].sum()
gb.mass = gb.mass.map(lambda x: th.round_sig(x,3))
gb

In [ ]:
# print(len(dgb))

In [ ]:
well_gb = ws_df.groupby('DisclosureId',as_index=False)[['date','OperatorName','api10','APINumber',
                                                        'bgLatitude','bgLongitude','ingKeyPresent',
                                                        'TotalBaseWaterVolume']].first()
well_gb['year'] = well_gb.date.dt.year
# well_gb = well_gb.drop('DisclosureId',axis=1)
well_gb = well_gb.sort_values('date')
display(maps.create_integrated_point_map(data=well_gb,area_df=containing_watershed,
                                         filled_area_df=containing_watershed,
                                         include_filled_shape=True,
                                        height=600,width=600))


In [ ]:
show_water_used(well_gb)

In [ ]:
chem_obj = create_chem_summary(ws_df)
show_chem_summary(chem_obj)


In [ ]:
import datetime
this_year = datetime.datetime.today().year

import seaborn as sns
import matplotlib.pyplot as plt
import io
import base64

def mass_plot_html(cas,df):
    c = df.CASNumber==cas
    df['year'] = df.date.dt.year
    gb = df[c].groupby('year',as_index=False)['mass'].sum()
    yrs = pd.DataFrame({'year':range(2011,this_year+1)})
    # print(yrs)
    gb = gb.merge(yrs,how='outer')
    gb.mass = gb.mass.fillna(0)
    
    sns.set_theme(style="white")
    plt.figure(figsize=(2, .75))
    ax = sns.barplot(data=gb, x='year', y='mass', errorbar=None,width=1)
    
    # Add titles and labels
    # ax.set_title('Mass by Year')
    ax.set_xlabel('')
    ax.set_ylabel('')
    # gb.plot.bar('year','mass',style='o',legend=False,
    #            xlim =(2011,ws_df.year.max()))
    
    years = sorted(gb['year'].unique())
    ticks = range(len(years)) # Positions: 0, 1, 2, ...
    labels = [str(year) if i == 0 or i == len(years) - 1 else "" for i, year in enumerate(years)]
    
    # 2. Set the locations first (This clears the warning)
    ax.set_xticks(ticks)
    
    # 3. Set the labels
    ax.set_xticklabels(labels,fontsize=8);
    ax.set_yticklabels([]);
    massmax = gb.mass.max()
    massmax_str = th.round_sig(massmax,3)
    maxyear = gb[gb.mass==massmax].year.tolist()[0]
    # ax.text(1.05, 0.5, f'Max mass ({massmax_str} pounds) in {gb[gb.mass==massmax].year.tolist()[0]}', 
    #         transform=ax.transAxes, 
    #         verticalalignment='center',
    #         fontsize=11,
    #         bbox=dict(facecolor='none', edgecolor=None, pad=5.0)); # Optional box for flair


    sns.despine()
    
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close() # Close plot to prevent it showing up twice in notebooks
    
    # 3. Encode the buffer to Base64
    data = base64.b64encode(buf.getbuffer()).decode("ascii")
    
    # 4. Return the HTML string
    return f'<img src="data:image/png;base64,{data}" width="150">',massmax_str, maxyear

caslst = []
masslst = []
yearlst = []
figlst = []

for cas in ws_df.CASNumber.unique().tolist():
    caslst.append(cas)
    img,massmax,maxyear = mass_plot_html(cas,ws_df)
    masslst.append(massmax)
    yearlst.append(maxyear)
    figlst.append(img)

out = pd.DataFrame({'cas':caslst,'max_year_pounds':masslst,'max_year':yearlst,'fig_mass_year':figlst})
out

In [ ]:
import openFF.common.generate_PDF_report_v1 as gPDF
outdir = r"C:\MyDocs\integrated\gwa_local\tools\tmp"
outfn = os.path.join(outdir,'PDF_report.pdf')

report_meta = {
    'report_name': "Watershed Report",
    'Watershed Scale': huc_scale,
    'Name': watershed_name,
    'Target latitude': latitude,
    'Target longitude': longitude,
}

report = gPDF.PDFReport(outfn)
report.build_title_page('Watershed Report',intro_paragraph='Test paragraph\n\nStil more',
                       meta_info=report_meta)
report.build_well_list(well_gb[['date','OperatorName','api10','TotalBaseWaterVolume']])
report.build_water_graphic(os.path.join(outdir,'water_use.jpg'))
report.build_chemical_summary(chem_obj.get_display_table(colset='pdf_report1'))
report.generate()
